# LFM2.5-2.6B · Stage 1: serving + harness compatibility

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neumbilly/NextSearch/blob/cursor/lfm2.5-2.6b-stage1-3be6/notebooks/01_lfm_serving_and_harness.ipynb)

Run **Stage 1** of the LFM2.5-2.6B web-research experiment end to end on a fresh
Colab **GPU** runtime: serve the model with vLLM, prove its native tool calling
works with the NextSearch harness, run reproducible smoke and development
rollouts against Parallel Turbo, and view telemetry and training curves
**live, inline — no external tracker (no W&B)**.

**Secrets** (add these in the Colab *Secrets* panel, key icon on the left):
`PARALLEL_API_KEY` (search/fetch), `HF_TOKEN` (model + dataset downloads),
and `GEMINI_API_KEY` (only for the optional grading cell — the judge runs on
Gemini's OpenAI-compatible endpoint).

**Architecture boundary** — vLLM does rendering/generation/tool-parsing; the
NextSearch *harness* owns prompts, tools, the episode loop and budgets; the
*evaluation* stack does prepare/rollout/grade/report; the *experiment* layer
(`nextsearch.experiment`) does telemetry + a local `RunLogger` + this live
viewer. This notebook is thin orchestration only. Full protocol and Stage-1
acceptance criteria: `docs/lfm2-step1.md`.

**No GPU?** vLLM will not start — the CPU-safe cells (install, telemetry, the
viewer) still work; GPU-only cells are labelled.

## 1 · Clone the repo and install it (editable)

In [ ]:
import os
if not os.path.isdir("NextSearch"):
    !git clone https://github.com/neumbilly/NextSearch.git
%cd NextSearch
# Stage-1 lives on this branch until it merges to main.
!git fetch --all --quiet && git checkout cursor/lfm2.5-2.6b-stage1-3be6 --quiet
# Editable install with the optional experiment extra (matplotlib + IPython for
# the live viewer). Core NextSearch never requires these.
!pip install --quiet -e ".[experiment]"
print("installed nextsearch from", os.getcwd())

## 2 · Install vLLM

LFM2.5's architecture and the `lfm2` tool-call parser ship in **vLLM >= 0.23.0**.
This wheel is large; on a fresh runtime it takes a few minutes.

In [ ]:
!pip install --quiet "vllm>=0.23.0"

## 3 · Load your Colab secrets into the environment

In [ ]:
import os
from google.colab import userdata

for key in ("PARALLEL_API_KEY", "HF_TOKEN", "GEMINI_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: loaded")
    except Exception as e:
        print(f"{key}: NOT set — {e}")

# vLLM / huggingface_hub read this variable name for gated or rate-limited pulls.
if os.environ.get("HF_TOKEN"):
    os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", os.environ["HF_TOKEN"])

## 4 · Hardware and versions (a)

In [ ]:
import platform, subprocess, sys

def _v(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return "not installed"

print("Python :", platform.python_version())
print("Torch  :", _v("torch"))
print("vLLM   :", _v("vllm"))
try:
    import torch
    print("CUDA   :", torch.version.cuda, "| available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"GPU    : {p.name} | {p.total_memory/1e9:.1f} GB VRAM")
    else:
        print("GPU    : none — vLLM cells will not run; CPU-safe cells still work")
except Exception as e:
    print("torch not importable yet:", e)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader 2>/dev/null || echo "nvidia-smi unavailable"

## 5 · Configuration — edit here

Every knob for the run lives in this one cell.

In [ ]:
CFG = {
    # repo / model identity
    "model_id": "LiquidAI/LFM2.5-2.6B",     # HF id, sent on the wire
    "served_model_name": "LiquidAI/LFM2.5-2.6B",  # must match model_id
    "registry_name": "lfm2.5-2.6b",         # entry in nextsearch.models.registry
    # serving (vLLM)
    "context_window": 32768,                # --max-model-len (start here)
    "gpu_memory_utilization": 0.90,
    "max_num_seqs": 32,                     # start here
    "tool_parser": "lfm2",
    "reasoning_parser": None,               # set "qwen3" for LFM2.5 reasoning models
    "port": 8000,
    # harness
    "harness_cap": 28000,                   # policy context cap, BELOW the window
    "task_date": "2026-07-31",              # frozen date placed in the prompt
    # experiment
    "experiment_name": "lfm2.5-2.6b",
    "dev_rollout_n": 20,
    "gpu_hourly_usd": 0.72,                 # ~L4 on-demand; used for $/episode
    "gpu_label": "L4",
    # persistence — a Drive path survives Colab disconnects; falls back to
    # /content (ephemeral) if Drive is not mounted.
    "output_root": "/content/drive/MyDrive/nextsearch-lfm/runs",
    "mount_drive": True,
}
BASE_URL = f"http://localhost:{CFG['port']}/v1"
os.environ["NEXTSEARCH_BASE_URL"] = BASE_URL
CFG

## 6 · Start vLLM  ⚠️ GPU only

The server is launched with `subprocess.Popen` (the handle is kept and an
`atexit` hook is registered — not `nohup`), logs stream to a file, and a bounded
readiness loop polls `/v1/models`. If startup fails, the server log is printed.

In [ ]:
import atexit, subprocess, time, urllib.request

VLLM_LOG = "/content/vllm.log"
_server = {"proc": None}

def start_vllm():
    if _server["proc"] and _server["proc"].poll() is None:
        print("vLLM already running (pid", _server["proc"].pid, ")"); return
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No CUDA GPU detected — vLLM cannot serve LFM2.5 on CPU. In Colab: "
            "Runtime > Change runtime type > Hardware accelerator > GPU "
            "(T4/L4/A100), then re-run the notebook from cell 1 (changing the "
            "runtime wipes the installs).")
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", CFG["model_id"],
        "--served-model-name", CFG["served_model_name"],
        "--max-model-len", str(CFG["context_window"]),
        "--gpu-memory-utilization", str(CFG["gpu_memory_utilization"]),
        "--max-num-seqs", str(CFG["max_num_seqs"]),
        "--enable-auto-tool-choice", "--tool-call-parser", CFG["tool_parser"],
        "--port", str(CFG["port"]),
    ]
    if CFG["reasoning_parser"]:
        cmd += ["--reasoning-parser", CFG["reasoning_parser"]]
    log = open(VLLM_LOG, "w")
    print("launching:", " ".join(cmd))
    _server["proc"] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    atexit.register(stop_vllm)

def stop_vllm():
    proc = _server.get("proc")
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
    _server["proc"] = None

def wait_ready(timeout_s=900, interval_s=5):
    url = f"{BASE_URL}/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if _server["proc"] and _server["proc"].poll() is not None:
            print("vLLM exited early — last 60 log lines:")
            print("".join(open(VLLM_LOG).readlines()[-60:]))
            raise RuntimeError("vLLM process exited during startup")
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    print("vLLM ready at", url); return True
        except Exception:
            pass
        time.sleep(interval_s)
    print("timed out — last 60 log lines:")
    print("".join(open(VLLM_LOG).readlines()[-60:]))
    raise TimeoutError(f"vLLM not ready after {timeout_s}s")

start_vllm()
wait_ready()

## 7 · Compatibility gate — `nextsearch.compat`

A synthetic-tool probe: **no `PARALLEL_API_KEY`, no search credits, two model
calls.** It confirms the server emits OpenAI-format tool calls the harness can
parse, dispatch, and feed back. Stop here if it fails — every rollout would be
zeros.

In [ ]:
import json
from nextsearch.compat import run_probe

result = run_probe(model_name=CFG["registry_name"], base_url=BASE_URL)
print(json.dumps(result, indent=2))
assert result["passed"], "compatibility gate FAILED — see checks above"
print("\nreasoning_content captured:", result["reasoning_content_captured"])

## 8 · Persistent run directory + `RunLogger`

Optionally mount Google Drive, then mint a **unique** run directory and point
`NEXTSEARCH_HOME` at it so every artifact (manifests, rollouts, telemetry,
metrics, and future adapter checkpoints) lands under one persistent root. Nothing
is ever overwritten.

In [ ]:
if CFG["mount_drive"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted (using ephemeral /content):", e)
        CFG["output_root"] = "/content/nextsearch-lfm/runs"

import vllm
from nextsearch.experiment.runlog import RunLogger

log = RunLogger(CFG["output_root"], experiment=CFG["experiment_name"], config={
    **CFG, "base_url": BASE_URL, "gpu": CFG["gpu_label"],
    "vllm_version": vllm.__version__,
})
RUN_DIR = log.dir
os.environ["NEXTSEARCH_HOME"] = str(RUN_DIR)   # datasets/ and runs/ live here
print("run dir:", RUN_DIR)

## 9 · Prepare SEAL-0

Pulls the source dataset and applies the audited gold revisions, writing
canonical rows + a frozen manifest under `NEXTSEARCH_HOME/datasets/`.

In [ ]:
from nextsearch import benchmarks

manifest = benchmarks.get("seal0").prepare()
print("seal0 prepared:", manifest["n_rows"], "rows, revision", manifest.get("revision"))

## 10 · Smoke test — one ungraded live rollout via the harness modules

Runs a single real research episode (live Parallel search + the served model)
straight through `run_episode`, and prints the answer, turns, stop reason, and
cost. No judge, no persistence — just proof the loop works end to end.

In [ ]:
import asyncio
from nextsearch import benchmarks, harnesses
from nextsearch.harness import run_episode
from nextsearch.models import get_client, get_model

harness = harnesses.get("solo")
model = get_model(CFG["registry_name"], base_url=BASE_URL)
row = benchmarks.get("seal0").load_rows(1)[0]

rollout = asyncio.run(run_episode(
    get_client(model), model, row, harness.tools(),
    max_turns=10, max_context=CFG["harness_cap"],
    system_suffix=harness.system_suffix(CFG["task_date"]), bench="seal0"))

print("Q:", row.meta.get("question") or row.messages[-1]["content"][:200])
print("\nANSWER:\n", (rollout.messages[-1]["content"] or "")[:2000])
print("\nturns:", rollout.n_turns, "| stop:", rollout.meta["stop_reason"],
      "| cost $:", rollout.timing.get("cost_usd"),
      "| wall s:", rollout.timing.get("wall_s"))

## 11 · Development rollout (configurable N, persisted)

Runs `CFG["dev_rollout_n"]` SEAL-0 tasks through the evaluation pipeline, which
appends a `rollouts.jsonl` under the run directory (resumable, one line per
episode). This is the artifact telemetry and the viewer read.

In [ ]:
import subprocess, sys

eval_id_file = RUN_DIR / "last_eval_id.txt"
cmd = [sys.executable, "-m", "nextsearch.cli", "rollout",
       "--benches", f"seal0:{CFG['dev_rollout_n']}",
       "--models", CFG["registry_name"],
       "--base-url", BASE_URL,
       "--date", CFG["task_date"]]
print("running:", " ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True, env=os.environ)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print(proc.stderr[-4000:]); raise RuntimeError("rollout failed")

## 12 · Locate and display a rollout JSON

In [ ]:
import json
from pathlib import Path

rollout_files = sorted(Path(RUN_DIR).rglob("rollouts.jsonl"))
print("rollout files:")
for f in rollout_files:
    print(" ", f, "(", sum(1 for _ in open(f)), "episodes )")

first = json.loads(open(rollout_files[-1]).readline())
print("\nfirst episode:", first["sample_id"], "| stop:",
      first["meta"]["stop_reason"], "| turns:", first["n_turns"])
print(json.dumps({k: first[k] for k in ("sample_id", "model", "n_turns",
      "n_tokens", "truncated")}, indent=2))

## 13 · Telemetry — `nextsearch.experiment`

Extract per-episode + aggregate telemetry (behavior, tokens, latency,
throughput, cost), write JSON + CSV under the run dir, and log the aggregate to
`metrics.jsonl` so it appears in the live view.

In [ ]:
import json
from nextsearch.experiment import aggregate, rollout_telemetry

episodes = []
for f in sorted(Path(RUN_DIR).rglob("rollouts.jsonl")):
    episodes += rollout_telemetry(f, gpu=CFG["gpu_label"],
                                  vllm_version=vllm.__version__,
                                  gpu_hourly_usd=CFG["gpu_hourly_usd"])
summary = aggregate(episodes, gpu_hourly_usd=CFG["gpu_hourly_usd"])
print(json.dumps(summary, indent=2))

# Persist telemetry beside the run and log the summary for the live view.
(RUN_DIR / "telemetry.json").write_text(json.dumps(
    {"aggregate": summary, "episodes": episodes}, indent=2))
log.log_summary(summary, phase="eval")

## 14 · Live experiment view + training curves (inline, no W&B)

`render` draws the dashboard once; `live_view` redraws it in place every few
seconds while a rollout streams (Colab stop button or `max_seconds` ends it).
The bottom-right panel shows **training curves** the moment any SFT/OPD/RL stage
logs `step` records to this run's `metrics.jsonl` — Stage 1 shows output-TPS
there instead.

In [ ]:
%matplotlib inline
from nextsearch.experiment.viewer import render, live_view

# One-shot snapshot:
fig = render(RUN_DIR, gpu=CFG["gpu_label"], gpu_hourly_usd=CFG["gpu_hourly_usd"],
             title=f"LFM2.5-2.6B · {RUN_DIR.name}")
import matplotlib.pyplot as plt; plt.show()

# Live view (uncomment to watch a rollout stream in another cell / re-run):
# live_view(RUN_DIR, refresh_s=5, gpu=CFG["gpu_label"],
#           gpu_hourly_usd=CFG["gpu_hourly_usd"], stop_when_idle_s=30)

## 15 · (Optional) Grade with Gemini + report

Grading is a **paid judge** step. You have `GEMINI_API_KEY`, so pass the Gemini
judge explicitly. Note: this is a *different* judge than the reported
NextSearch-1 numbers used, so scores here are self-consistent but not comparable
to the headline table.

In [ ]:
import subprocess, sys

eval_id = sorted(p.name for p in (RUN_DIR / "runs").iterdir())[-1]
for stage in (["grade", "--eval-id", eval_id, "--judge", "gemini-3.6-flash"],
              ["report", "--eval-id", eval_id]):
    proc = subprocess.run([sys.executable, "-m", "nextsearch.cli", *stage],
                          capture_output=True, text=True, env=os.environ)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print(proc.stderr[-3000:])

## 16 · Shut down vLLM cleanly

In [ ]:
stop_vllm()
print("vLLM stopped")